# Taller: Clasificación de Películas con Perceptrón

Curso: Inteligencia Artificial / Machine Learning

Duración estimada: 2–3 horas

Objetivo: Utilizar un **Perceptrón** para clasificar películas como **exitosas o no exitosas** usando el dataset de Kaggle **TMDB 5000 Movies**.

Dataset:
https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

Archivo principal: `tmdb_5000_movies.csv`

## 1. Introducción

El **Perceptrón** es uno de los algoritmos más antiguos de aprendizaje automático.

Este modelo intenta encontrar una **frontera de decisión lineal** que separe dos clases.

En este taller intentaremos clasificar películas según su éxito utilizando información del dataset.

Preguntas guía:

- ¿Las películas exitosas pueden separarse linealmente?
- ¿Qué características ayudan más a la clasificación?
- ¿Qué limitaciones tiene el perceptrón?

## 2. Carga del Dataset

Pasos:

1. Descargar el dataset desde Kaggle.
2. Cargar el archivo `tmdb_5000_movies.csv`.
3. Mostrar:
   - número de registros
   - número de columnas
   - nombres de las variables
   - tipos de datos

Utilizar **pandas** para explorar el dataset.

In [ ]:
# Carga del dataset y librerías base
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

data_path = Path('tmdb_5000_movies.csv')
if not data_path.exists():
    raise FileNotFoundError(f'No se encontró el archivo: {data_path.resolve()}')

df_raw = pd.read_csv(data_path)
print(f'Registros: {df_raw.shape[0]}')
print(f'Columnas: {df_raw.shape[1]}')
print('\nPrimeras columnas:')
print(df_raw.columns.tolist()[:15])
print('\nTipos de datos (primeras 15):')
display(df_raw.dtypes.head(15))

# Trabajaremos sobre una copia para no perder el dataframe original
df = df_raw.copy()

: 

## 3. Exploración de Variables

Seleccione variables numéricas relevantes.

Ejemplos:

- budget
- popularity
- runtime
- vote_average
- vote_count
- revenue

Analice:

- estadísticas descriptivas
- correlaciones
- distribución de variables

In [ ]:
# Exploración rápida
print('Dimensión del dataset:', df.shape)
df.head(5)

(4803, 20)

In [ ]:
# Nombres de columnas
pd.Series(df.columns, name='columnas')

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe(include=[np.number]).T

,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4.803000e+03,4803.000000,4803.000000,4.803000e+03,4801.000000,4803.000000,4803.000000
mean,2.904504e+07,57165.484281,21.492301,8.226064e+07,106.875859,6.092172,690.217989
std,4.072239e+07,88694.614033,31.816650,1.628571e+08,22.611935,1.194612,1234.585891
min,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,7.900000e+05,9014.500000,4.668070,0.000000e+00,94.000000,5.600000,54.000000
50%,1.500000e+07,14629.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000
75%,4.000000e+07,58610.500000,28.313505,9.291719e+07,118.000000,6.800000,737.000000
max,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000


## 4. Definición de la Variable Objetivo

Defina una variable llamada **success**.

Ejemplo de criterio:

- success = 1 si vote_average >= 7
- success = 0 en otro caso

También puede probar otros criterios como:

- revenue > budget

In [ ]:
# Definir variables numéricas relevantes
features = ['budget', 'popularity', 'runtime', 'vote_average', 'vote_count', 'revenue']

missing_features = [col for col in features if col not in df.columns]
if missing_features:
    raise ValueError(f'Faltan columnas en el dataset: {missing_features}')

df = df[features].copy()
df.head()

,budget,popularity,runtime,vote_average,vote_count,revenue
0,237000000,150.437577,162.0,7.2,11800,2787965087
1,300000000,139.082615,169.0,6.9,4500,961000000
2,245000000,107.376788,148.0,6.3,4466,880674609
3,250000000,112.312950,165.0,7.6,9106,1084939099
4,260000000,43.926995,132.0,6.1,2124,284139100


In [ ]:
# Criterio de éxito base: voto promedio >= 7
df['success'] = (df['vote_average'] >= 7).astype(int)

print('Distribución de la variable success:')
display(df['success'].value_counts().rename('conteo').to_frame())
display(df.head())

,budget,popularity,runtime,vote_average,vote_count,revenue,success
0,237000000,150.437577,162.0,7.2,11800,2787965087,1
1,300000000,139.082615,169.0,6.9,4500,961000000,0
2,245000000,107.376788,148.0,6.3,4466,880674609,0
3,250000000,112.312950,165.0,7.6,9106,1084939099,1
4,260000000,43.926995,132.0,6.1,2124,284139100,0


## 5. Limpieza de Datos

Realizar:

- eliminación de valores faltantes
- eliminación de columnas irrelevantes
- selección de features
- normalización o estandarización de variables

Preguntas:

- ¿Cuántos registros quedan después de limpiar el dataset?
- ¿Qué variables parecen más informativas?

In [ ]:
# Limpieza y normalización
from sklearn.preprocessing import StandardScaler

print('Valores faltantes por columna (antes):')
display(df.isna().sum().to_frame('faltantes'))

df_clean = df.dropna().drop_duplicates().copy()
print(f'Registros después de limpiar: {df_clean.shape[0]}')

X_cols = ['budget', 'popularity', 'runtime', 'vote_average', 'vote_count', 'revenue']
X = df_clean[X_cols].copy()
y = df_clean['success'].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_cols, index=X.index)

print('Resumen de features escaladas:')
display(X_scaled_df.describe().T)

# Correlaciones para discutir variables informativas
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[X_cols + ['success']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

## 6. Análisis de Separabilidad Lineal

Antes de entrenar el perceptrón debemos analizar si las clases parecen **linealmente separables**.

Actividades:

1. Seleccionar dos variables.
2. Construir un gráfico de dispersión.
3. Colorear por clase.

Probar varias combinaciones:

- vote_average vs popularity
- vote_average vs vote_count
- budget vs revenue
- runtime vs popularity

Preguntas:

- ¿Se observa una frontera aproximadamente lineal?
- ¿Las clases están mezcladas?

In [ ]:
# Análisis de separabilidad lineal con dos variables
x_var, y_var = 'vote_average', 'popularity'

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_clean.sample(min(2000, len(df_clean)), random_state=42),
    x=x_var,
    y=y_var,
    hue='success',
    alpha=0.7,
    palette='Set1'
 )
plt.title(f'Separabilidad lineal: {x_var} vs {y_var}')
plt.legend(title='success')
plt.tight_layout()
plt.show()

print('Prueba visual sugerida adicional: vote_average vs vote_count, budget vs revenue, runtime vs popularity')

## 7. Entrenamiento del Perceptrón

Utilizar **scikit-learn** para entrenar el modelo.

Pasos:

1. Separar datos en **train** y **test**.
2. Crear el modelo Perceptrón.
3. Entrenar el modelo.
4. Realizar predicciones.

In [ ]:
# Entrenamiento del Perceptrón
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.25, random_state=42, stratify=y
)

perceptron = Perceptron(max_iter=2000, eta0=0.01, random_state=42)
perceptron.fit(X_train, y_train)

y_pred = perceptron.predict(X_test)
print('Modelo entrenado correctamente.')

## 8. Evaluación del Modelo

Evaluar el modelo utilizando:

- accuracy
- precision
- recall
- matriz de confusión

Analice:

- ¿El modelo clasifica bien?
- ¿Qué errores comete?

In [ ]:
# Evaluación del modelo
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión - Perceptrón')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## 9. Frontera de Decisión

Seleccione dos variables y visualice:

- datos
- frontera de decisión

Analice:

- puntos mal clasificados
- posición de la frontera

In [ ]:
# Frontera de decisión con 2 variables
from sklearn.preprocessing import StandardScaler

vars_2d = ['vote_average', 'popularity']
X2 = df_clean[vars_2d].copy()
y2 = df_clean['success'].copy()

scaler_2d = StandardScaler()
X2_scaled = scaler_2d.fit_transform(X2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2_scaled, y2, test_size=0.25, random_state=42, stratify=y2
)

clf_2d = Perceptron(max_iter=2000, eta0=0.01, random_state=42)
clf_2d.fit(X2_train, y2_train)

x_min, x_max = X2_scaled[:, 0].min() - 1, X2_scaled[:, 0].max() + 1
y_min, y_max = X2_scaled[:, 1].min() - 1, X2_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = clf_2d.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
sns.scatterplot(
    x=X2_scaled[:, 0],
    y=X2_scaled[:, 1],
    hue=y2,
    palette='Set1',
    alpha=0.7,
    edgecolor=None
)
plt.title('Frontera de decisión (Perceptrón 2D)')
plt.xlabel(f'{vars_2d[0]} (escalada)')
plt.ylabel(f'{vars_2d[1]} (escalada)')
plt.legend(title='success')
plt.tight_layout()
plt.show()

y2_pred = clf_2d.predict(X2_test)
print(f'Accuracy modelo 2D: {accuracy_score(y2_test, y2_pred):.4f}')

## 10. Experimentos

Realizar los siguientes experimentos:

Experimento 1
- usar solo 2 variables

Experimento 2
- usar 4 variables

Experimento 3
- cambiar la definición de éxito

Experimento 4
- eliminar variables irrelevantes

Comparar resultados.

In [ ]:
# Experimentos comparativos
from sklearn.metrics import accuracy_score, precision_score, recall_score

def run_experiment(dataframe, feature_cols, success_mode='vote_average'):
    data = dataframe.copy()
    if success_mode == 'vote_average':
        data['success'] = (data['vote_average'] >= 7).astype(int)
    elif success_mode == 'revenue_budget':
        data['success'] = (data['revenue'] > data['budget']).astype(int)
    else:
        raise ValueError('success_mode no soportado')

    cols_needed = list(set(feature_cols + ['success']))
    exp_df = data[cols_needed].dropna().drop_duplicates()
    X_exp = exp_df[feature_cols]
    y_exp = exp_df['success']

    if y_exp.nunique() < 2:
        return None

    scaler_exp = StandardScaler()
    X_exp_scaled = scaler_exp.fit_transform(X_exp)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_exp_scaled, y_exp, test_size=0.25, random_state=42, stratify=y_exp
    )

    model = Perceptron(max_iter=2000, eta0=0.01, random_state=42)
    model.fit(X_tr, y_tr)
    y_hat = model.predict(X_te)

    return {
        'n_features': len(feature_cols),
        'features': ', '.join(feature_cols),
        'success_mode': success_mode,
        'accuracy': accuracy_score(y_te, y_hat),
        'precision': precision_score(y_te, y_hat, zero_division=0),
        'recall': recall_score(y_te, y_hat, zero_division=0),
        'n_rows': len(exp_df)
    }

experiments = [
    ('Experimento 1: 2 variables', ['vote_average', 'popularity'], 'vote_average'),
    ('Experimento 2: 4 variables', ['budget', 'runtime', 'vote_average', 'vote_count'], 'vote_average'),
    ('Experimento 3: éxito alternativo', ['budget', 'popularity', 'runtime', 'revenue'], 'revenue_budget'),
    ('Experimento 4: sin budget (posible irrelevante)', ['popularity', 'runtime', 'vote_average', 'vote_count'], 'vote_average'),
]

results = []
for name, feats, mode in experiments:
    out = run_experiment(df_raw, feats, mode)
    if out is not None:
        out['experimento'] = name
        results.append(out)

results_df = pd.DataFrame(results)[['experimento', 'n_rows', 'n_features', 'features', 'success_mode', 'accuracy', 'precision', 'recall']]
results_df = results_df.sort_values('accuracy', ascending=False).reset_index(drop=True)
results_df

## 11. Discusión

Responder:

1. ¿El dataset parece linealmente separable?
2. ¿Qué variables ayudan más al perceptrón?
3. ¿Qué limitaciones tiene este modelo?
4. ¿Qué algoritmo podría resolver mejor este problema?

Opciones posibles:

- SVM
- Árboles de decisión
- Redes neuronales

## 12. Conclusiones

Escriba un breve análisis final donde explique:

- qué aprendió del perceptrón
- si el problema es linealmente separable
- qué mejoras podrían hacerse